## 其他中间件


In [4]:
from dotenv import load_dotenv
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, ModelFallbackMiddleware, \
    LLMToolSelectorMiddleware, ToolRetryMiddleware, ModelRetryMiddleware, LLMToolEmulator, ContextEditingMiddleware, \
    ClearToolUsesEdit, FilesystemFileSearchMiddleware
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
)

agent = create_agent(
    model = model,
    middleware=[
        ModelCallLimitMiddleware(# 模型调用限制中间件
            thread_limit=3, # 每个线程最多3次模型调用
            run_limit=3, # 每次运行最多3次
            exit_behavior="end", # 达到限制后退出/error抛异常
        ),
        ToolCallLimitMiddleware(# 工具调用限制中间件
            #thread_limit=2,
            run_limit=5,
            exit_behavior="end",
        ),
        ModelFallbackMiddleware(# 备用模型中间件
                init_chat_model("deepseek:deepseek-v4-flash"),
                init_chat_model("deepseek:deepseek-v4-pro")
        ),
        LLMToolSelectorMiddleware(# 工具选择中间件
            model="deepseek:deepseek-v4-flash",# 工具选择模型
            max_tools=5, # 最多选择 5 个工具
            always_include=["get_weather"]# 常驻工具
        ),
        ToolRetryMiddleware(# 工具重试中间件
            max_retries=6, # 最大重试次数（不包含初始的那次调用，一共最多调 1 + 6 =7 次）
            backoff_factor=2.0, # 指数退避因子（每次重试等待时间乘以 2）
            initial_delay=1.0, # 第一次重试前的初始等待时间（1 秒）
            max_delay=10.0, # 最大等待延迟上限（防止指数增长无限大，限制在 10 秒）
            jitter=True, # 开启抖动（在等待时间中加入随机性，防止并发请求时出现“惊群效应”）
            retry_on=(TimeoutError,), # 仅针对捕获到特定的 TimeoutError 异常时才触发重试
            on_failure="continue" # 当达到最大重试次数依然失败时，Agent 的行为："continue" 表示将错误信息包装后塞回对话历史，让大模型知道失败了并继续决策
        ),
        ModelRetryMiddleware(# 模型重试中间件
            max_retries=6,
            backoff_factor=2.0,
            initial_delay=1.0,
            max_delay=10.0,
            on_failure="error",
            jitter=False,
        ),
        LLMToolEmulator(# 工具模拟中间件,模型帮你虚拟回复工具调用结果
            model=model,
        ),
        ContextEditingMiddleware(# 上下文编辑中间件
            edits=[
                ClearToolUsesEdit(
                    trigger=50,#工具调用次数超过50次后，清除工具调用记录
                    keep=0,# 保留最近0次工具调用记录
                ),
            ],
        ),
        FilesystemFileSearchMiddleware(# 文件系统文件搜索中间件
            root_path="../todo_workspace", #搜索目录
            # 是否启用 ripgrep 搜索引擎：
            # 设为 True 可以获得比原生 Grep 更快的性能（前提是系统已安装 ripgrep）
            use_ripgrep=True,
            # 单个文件的最大读取限制（单位MB）：防止读取超大型日志或二进制文件导致 OOM
            max_file_size_mb=10
        ),
    ],
)
